In [29]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd
from pandas import read_csv
from joblib import dump
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error


mlflow.set_tracking_uri("http://localhost:5555")


In [30]:
# Load  dataset
df = pd.read_csv("asteroids_data.csv")

# Keep only columns needed for ML
df = df[["magnitude", "aphelion_distance"]]

# Convert columns to numeric
df["magnitude"] = pd.to_numeric(df["magnitude"], errors="coerce")
df["aphelion_distance"] = pd.to_numeric(df["aphelion_distance"], errors="coerce")

# Remove NaN ONLY in these columns
df.dropna(subset=["magnitude", "aphelion_distance"], inplace=True)

# Check size
print(df.shape)

(2000, 2)


In [31]:
# Import the function used to split data into training and testing sets
from sklearn.model_selection import train_test_split

# Select the input feature (independent variable)
# Here, "magnitude" is used to predict asteroid distance
X = df[["magnitude"]]

# Select the target/output variable (dependent variable)
# "aphelion_distance" is the value we want the model to predict
y = df["aphelion_distance"]

# Split the dataset into training and testing data
# test_size=0.2 means 20% of the data is used for testing
# and 80% is used for training
# random_state=42 ensures reproducible results
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Display the shape of the training dataset
# (number of rows and columns)
print(X_train.shape)

# Display the shape of the testing dataset
print(X_test.shape)

(1600, 1)
(400, 1)


In [32]:
with mlflow.start_run():


    mind = LinearRegression()
    mind.fit(X_train,  y_train)
    #Evaluate
    predictions= mind.predict(X_test)
    r2 = r2_score(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    
    #log Metrics

    mlflow.log_metric("r2_score", r2)
    mlflow.log_metric("mse", mse)

    #Log Model & Register

    result = mlflow.sklearn.log_model(sk_model=mind, artifact_path="model")

    mlflow.register_model(
            model_uri=result.model_uri,
            name ="my_linear_model"
            )
    print(f"Model logged with r2: {r2}")


with open("model.pkl", "wb") as file:
    pickle.dump(mind, file)

MlflowException: API request to http://localhost:5555/api/2.0/mlflow/runs/create failed with exception HTTPConnectionPool(host='localhost', port=5555): Max retries exceeded with url: /api/2.0/mlflow/runs/create (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000017C55BD79D0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))